In [1]:
import json
import pandas as pd
import numpy as np
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, make_scorer, accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve, cohen_kappa_score, roc_auc_score, confusion_matrix, plot_precision_recall_curve, plot_roc_curve, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier, plot_importance
import tensorflow as tf

# FCPython is a set of functions distributed as part of the code for Uppsala University's 
# Mathematical Modelling of Football course (https://uppsala.instructure.com/courses/28112)
# which was produced based on the content put out by FCPython (https://fcpython.com/)
from FCPython import createPitch

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

## Load one season of LaLiga data

The Statsbomb data contains 15 seasons of La Liga data where Lionel Messi plays, spanning from 2019/20 back to 2004/2005.

I'll start by loading one season of data, having a look at it, before moving on to using all 15 seasons for my model

In [3]:
with open('C:/Users/matth/Google Drive/Data Science/Football Analytics/SoccermaticsForPython-master/Statsbomb/data/competitions.json') as f:
    competitions = json.load(f)
    competitions = pd.json_normalize(competitions)
competitions[competitions.competition_name == 'La Liga'].head()

,competition_id,season_id,country_name,competition_name,competition_gender,season_name,match_updated,match_available
19,11,42,Spain,La Liga,male,2019/2020,2020-09-04T15:17:40.862314,2020-09-04T15:17:40.862314
20,11,4,Spain,La Liga,male,2018/2019,2020-07-29T05:00,2020-07-29T05:00
21,11,1,Spain,La Liga,male,2017/2018,2020-07-29T05:00,2020-07-29T05:00
22,11,2,Spain,La Liga,male,2016/2017,2020-09-04T15:17:40.862314,2020-09-04T15:17:40.862314
23,11,27,Spain,La Liga,male,2015/2016,2020-07-29T05:00,2020-07-29T05:00


In [4]:
competition_id=11
season_ids = [37]
#Load the list of matches for this competition

all_games = pd.DataFrame(None)

for season_id in season_ids:
    with open(f'C:/Users/matth/Google Drive/Data Science/Football Analytics/SoccermaticsForPython-master/Statsbomb/data/matches/{str(competition_id)}/{str(season_id)}.json', encoding='utf-8') as f:
        matches = json.load(f)
        season_json = pd.json_normalize(matches, sep = "_")
        all_games = pd.concat([all_games,season_json],axis=0)
    
print(all_games.shape)
all_games.head()

(7, 38)


,match_id,match_date,kick_off,home_score,away_score,match_status,last_updated,match_week,competition_competition_id,competition_country_name,competition_competition_name,season_season_id,season_season_name,home_team_home_team_id,home_team_home_team_name,home_team_home_team_gender,home_team_home_team_group,home_team_country_id,home_team_country_name,away_team_away_team_id,away_team_away_team_name,away_team_away_team_gender,away_team_away_team_group,away_team_country_id,away_team_country_name,metadata_data_version,metadata_shot_fidelity_version,metadata_xy_fidelity_version,competition_stage_id,competition_stage_name,stadium_id,stadium_name,stadium_country_id,stadium_country_name,referee_id,referee_name,referee_country_id,referee_country_name
0,68314,2004-12-04,20:00:00.000,4,0,available,2020-07-29T05:00,14,11,Spain,La Liga,37,2004/2005,217,Barcelona,male,None,214,Spain,223,Málaga,male,None,214,Spain,1.1.0,2,2,1,Regular Season,342,Camp Nou,214,Spain,993,José Omar Losantos,NaN,NaN
1,68313,2004-10-24,21:00:00.000,3,0,available,2020-07-29T05:00,8,11,Spain,La Liga,37,2004/2005,217,Barcelona,male,None,214,Spain,422,Osasuna,male,None,214,Spain,1.1.0,2,2,1,Regular Season,342,Camp Nou,214,Spain,994,Vicente José Lizondo Cortés,NaN,NaN
2,68316,2005-05-01,19:00:00.000,2,0,available,2020-07-29T05:00,34,11,Spain,La Liga,37,2004/2005,217,Barcelona,male,None,214,Spain,608,Albacete,male,None,214,Spain,1.1.0,2,2,1,Regular Season,342,Camp Nou,214,Spain,407,Carlos Velasco Carballo,214.0,Spain
3,68315,2004-12-21,20:00:00.000,2,1,available,2020-07-29T05:00,17,11,Spain,La Liga,37,2004/2005,217,Barcelona,male,None,214,Spain,221,Levante,male,None,214,Spain,1.1.0,2,2,1,Regular Season,342,Camp Nou,214,Spain,222,David Fernández,112.0,Italy
4,69153,2004-12-11,20:00:00.000,1,2,available,2020-07-29T05:00,15,11,Spain,La Liga,37,2004/2005,608,Albacete,male,None,214,Spain,217,Barcelona,male,None,214,Spain,1.1.0,2,2,1,Regular Season,4448,Estadio Carlos Belmonte,214,Spain,1007,Alfonso Perez Burrull,NaN,NaN


In [5]:
# Pull out the Match IDs that I need and use these to load the matche events
chosen_ids = all_games['match_id']

for id in chosen_ids:
    with open("C:/Users/matth/Google Drive/Data Science/Football Analytics/SoccermaticsForPython-master/Statsbomb/data/events/"+str(id)+".json", encoding='utf-8') as f:
        events = json.load(f)
        all_matches = pd.json_normalize(events, sep = "_")
all_matches.head()

,id,index,period,timestamp,minute,second,possession,duration,type_id,type_name,possession_team_id,possession_team_name,play_pattern_id,play_pattern_name,team_id,team_name,tactics_formation,tactics_lineup,related_events,location,player_id,player_name,position_id,position_name,pass_recipient_id,pass_recipient_name,pass_length,pass_angle,pass_height_id,pass_height_name,pass_end_location,pass_type_id,pass_type_name,pass_body_part_id,pass_body_part_name,carry_end_location,under_pressure,pass_outcome_id,pass_outcome_name,ball_receipt_outcome_id,ball_receipt_outcome_name,pass_switch,duel_outcome_id,duel_outcome_name,duel_type_id,duel_type_name,off_camera,pass_cross,goalkeeper_outcome_id,goalkeeper_outcome_name,goalkeeper_type_id,goalkeeper_type_name,clearance_left_foot,clearance_body_part_id,clearance_body_part_name,ball_recovery_recovery_failure,foul_won_defensive,clearance_head,dribble_outcome_id,dribble_outcome_name,counterpress,out,pass_aerial_won,clearance_right_foot,clearance_aerial_won,pass_assisted_shot_id,pass_shot_assist,shot_statsbomb_xg,shot_end_location,shot_key_pass_id,shot_outcome_id,shot_outcome_name,shot_type_id,shot_type_name,shot_body_part_id,shot_body_part_name,shot_technique_id,shot_technique_name,shot_first_time,shot_freeze_frame,goalkeeper_end_location,goalkeeper_position_id,goalkeeper_position_name,pass_goal_assist,shot_deflected,block_deflection,goalkeeper_technique_id,goalkeeper_technique_name,goalkeeper_body_part_id,goalkeeper_body_part_name,pass_outswinging,pass_technique_id,pass_technique_name,interception_outcome_id,interception_outcome_name,shot_aerial_won,foul_committed_advantage,foul_won_advantage,pass_inswinging,shot_one_on_one,pass_straight,dribble_overrun,foul_committed_type_id,foul_committed_type_name,foul_committed_card_id,foul_committed_card_name,pass_through_ball,substitution_outcome_id,substitution_outcome_name,substitution_replacement_id,substitution_replacement_name,pass_no_touch,pass_cut_back,ball_recovery_offensive
0,3b66ba10-464a-4fc4-b1f1-081384c134f0,1,1,00:00:00.000,0,0,1,0.000000,35,Starting XI,214,Espanyol,1,Regular Play,214,Espanyol,433.0,"[{'player': {'id': 26103, 'name': 'Idriss Carl...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,51278bda-3e4e-4887-9c39-04b7522b39d5,2,1,00:00:00.000,0,0,1,0.000000,35,Starting XI,214,Espanyol,1,Regular Play,217,Barcelona,433.0,"[{'player': {'id': 20176, 'name': 'Víctor Vald...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,99020799-c2fe-407d-b830-a93ff5eb6b06,3,1,00:00:00.000,0,0,1,0.000000,18,Half Start,214,Espanyol,1,Regular Play,217,Barcelona,NaN,NaN,[6b838dec-5b3f-4df8-9f0b-3f275ee03b0a],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6b838dec-5b3f-4df8-9f0b-3f275ee03b0a,4,1,00:00:00.000,0,0,1,0.000000,18,Half Start,214,Espanyol,1,Regular Play,214,Espanyol,NaN,NaN,[99020799-c2fe-407d-b830-a93ff5eb6b06],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [6]:
# Check how many rows we have
all_matches.shape

(2770, 114)

In [7]:
# See what event types we have
all_matches.type_name.unique()

array(['Starting XI', 'Half Start', 'Pass', 'Ball Receipt*', 'Carry',
       'Pressure', 'Dispossessed', 'Duel', 'Ball Recovery',
       'Foul Committed', 'Foul Won', 'Goal Keeper', 'Clearance',
       'Dribbled Past', 'Dribble', 'Shot', 'Block', 'Miscontrol',
       'Offside', 'Interception', 'Shield', 'Half End', 'Injury Stoppage',
       'Substitution', 'Tactical Shift'], dtype=object)

I noticed in the Statsbomb documentation that there's a column called Under Pressure. As an exercise to understand how this column is produced and to sensecheck my understanding of the data, I wanted to try and reconstruct this field myself.

In [8]:
# First filter out all passes into a new dataframe
passes = all_matches[all_matches.type_name == 'Pass']
print(passes.shape)

# Use pandas Explode to explode out the list of related_events IDs into individual rows, this will allow us to join 
# the passes dataframe onto the dataframe of all events for the matches and look at our pressures

passes = passes.explode('related_events')

# Perform the join (join is Inner by default)

passes_pressure = pd.merge(passes,all_matches[['id','type_name']],left_on = 'related_events',right_on='id')
passes_pressure.head()

(786, 114)


,id_x,index,period,timestamp,minute,second,possession,duration,type_id,type_name_x,possession_team_id,possession_team_name,play_pattern_id,play_pattern_name,team_id,team_name,tactics_formation,tactics_lineup,related_events,location,player_id,player_name,position_id,position_name,pass_recipient_id,pass_recipient_name,pass_length,pass_angle,pass_height_id,pass_height_name,pass_end_location,pass_type_id,pass_type_name,pass_body_part_id,pass_body_part_name,carry_end_location,under_pressure,pass_outcome_id,pass_outcome_name,ball_receipt_outcome_id,ball_receipt_outcome_name,pass_switch,duel_outcome_id,duel_outcome_name,duel_type_id,duel_type_name,off_camera,pass_cross,goalkeeper_outcome_id,goalkeeper_outcome_name,goalkeeper_type_id,goalkeeper_type_name,clearance_left_foot,clearance_body_part_id,clearance_body_part_name,ball_recovery_recovery_failure,foul_won_defensive,clearance_head,dribble_outcome_id,dribble_outcome_name,counterpress,out,pass_aerial_won,clearance_right_foot,clearance_aerial_won,pass_assisted_shot_id,pass_shot_assist,shot_statsbomb_xg,shot_end_location,shot_key_pass_id,shot_outcome_id,shot_outcome_name,shot_type_id,shot_type_name,shot_body_part_id,shot_body_part_name,shot_technique_id,shot_technique_name,shot_first_time,shot_freeze_frame,goalkeeper_end_location,goalkeeper_position_id,goalkeeper_position_name,pass_goal_assist,shot_deflected,block_deflection,goalkeeper_technique_id,goalkeeper_technique_name,goalkeeper_body_part_id,goalkeeper_body_part_name,pass_outswinging,pass_technique_id,pass_technique_name,interception_outcome_id,interception_outcome_name,shot_aerial_won,foul_committed_advantage,foul_won_advantage,pass_inswinging,shot_one_on_one,pass_straight,dribble_overrun,foul_committed_type_id,foul_committed_type_name,foul_committed_card_id,foul_committed_card_name,pass_through_ball,substitution_outcome_id,substitution_outcome_name,substitution_replacement_id,substitution_replacement_name,pass_no_touch,pass_cut_back,ball_recovery_offensive,id_y,type_name_y
0,798b3924-aefa-4091-a8d4-1e8d598e8ac1,5,1,00:00:01.285,0,1,2,1.201624,30,Pass,217,Barcelona,9,From Kick Off,217,Barcelona,NaN,NaN,91366daa-5095-48ba-abec-7b83ae964e45,"[61.0, 41.0]",15743.0,Henrik Larsson,23.0,Center Forward,25879.0,Ronaldo de Assis Moreira,11.237883,2.935471,1.0,Ground Pass,"[50.0, 43.3]",65.0,Kick Off,40.0,Right Foot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91366daa-5095-48ba-abec-7b83ae964e45,Ball Receipt*
1,d48f439a-cd1a-48f0-81c4-10b06b00972b,8,1,00:00:02.952,0,2,2,1.495727,30,Pass,217,Barcelona,9,From Kick Off,217,Barcelona,NaN,NaN,2309324c-51e6-4e79-8d6d-b8e5b791d726,"[49.7, 45.8]",25879.0,Ronaldo de Assis Moreira,21.0,Left Wing,20125.0,Carles Puyol i Saforcada,15.697452,2.450933,1.0,Ground Pass,"[37.6, 55.8]",NaN,NaN,38.0,Left Foot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2309324c-51e6-4e79-8d6d-b8e5b791d726,Ball Receipt*
2,91339857-ed39-4696-a1ab-71b2df487723,11,1,00:00:04.895,0,4,2,1.394332,30,Pass,217,Barcelona,9,From Kick Off,217,Barcelona,NaN,NaN,f516d440-6166-477c-bd0d-a3f143fe454e,"[38.6, 55.2]",20125.0,Carles Puyol i Saforcada,5.0,Left Center Back,25873.0,Oleguer Presas Renom,27.772290,-2.029613,1.0,Ground Pass,"[26.3, 30.3]",NaN,NaN,40.0,Right Foot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

For passes with a related 'Pressure' event, check what values we have populated in the 'Under_Pressure' column

In [9]:
passes_pressure[passes_pressure.type_name_y == 'Pressure'].under_pressure.unique()

array([True], dtype=object)

I have not checked the converse (i.e. what values under_pressure has when type_name_y != Pressure) because there can be multiple related events for a given single event. As an example, a pass could have related events of: Ball Receipt, Pressure, Tackle all for one event. 

As such, I'm happy to proceed using under_pressure without producing another feature of my own here.